# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the Mass Spectrometry dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata_obj = dataset.metadata

print(f"{getattr(metadata_obj, 'name')}: {getattr(metadata_obj, 'description')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We list all record sets (`@id`), their field `@id`s, and the columns they contain.

In [ ]:
# List all record sets and their fields by @id
record_sets = metadata_obj.record_sets

print(f"Number of record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"RecordSet @id: {rs.id}")
    print(f"  name: {getattr(rs, 'name', None)}")
    print(f"  description: {getattr(rs, 'description', None)}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    Field @id: {field.id}")
        print(f"      name: {getattr(field, 'name', None)}")
        print(f"      dataType: {getattr(field, 'data_type', None)}")
        print(f"      column: {getattr(field, 'column', None)}")
    print()

### Example Record Overview
Print several example records for a selected record set.

In [ ]:
# Choose the first record set for demonstration
first_record_set_id = record_sets[0].id if record_sets else None
if first_record_set_id is not None:
    print(f"\nExample records from record set: {first_record_set_id}\n")
    for i, row in enumerate(dataset.records(record_set=first_record_set_id)):
        print(row)
        if i >= 2: break
else:
    print("No record sets found in the dataset.")

## 3. Data Extraction
Load data from one or more record sets into DataFrames for analysis. All usages reference entities via their `@id`.

In [ ]:
dfs = {}
all_record_set_ids = [rs.id for rs in record_sets]

# Extract into pandas DataFrames
for rsid in all_record_set_ids:
    # It is possible some record sets have no records, so we handle errors gracefully
    try:
        records = list(dataset.records(record_set=rsid))
        if len(records) > 0:
            dfs[rsid] = pd.DataFrame(records)
            print(f"Extracted {len(records)} records for RecordSet {rsid}")
        else:
            print(f"No records found for RecordSet {rsid}")
    except Exception as e:
        print(f"Error extracting records for {rsid}: {e}")

if dfs:
    main_rs_id = list(dfs.keys())[0]
    print(f"\nAvailable columns in DataFrame for {main_rs_id}:")
    print(dfs[main_rs_id].columns.tolist())
    display(dfs[main_rs_id].head())
else:
    main_rs_id = None
    print("No non-empty record set DataFrames extracted.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, including filtering, normalization, and grouping. All field columns are referenced by their `@id`.

In [ ]:
import numpy as np

# We'll demonstrate using the first numeric field from the first record set
if main_rs_id:
    df = dfs[main_rs_id]
    # Try to find a numeric column
    numeric_field_id = None
    group_field_id = None
    for rs in record_sets:
        if rs.id == main_rs_id:
            for field in rs.fields:
                dt = getattr(field, 'data_type', None)
                if dt and ('Float' in dt or 'Number' in dt or 'Integer' in dt):
                    if field.column in df.columns:
                        numeric_field_id = field.column
                        break
            # Look for a categorical/group field:
            for field in rs.fields:
                dt = getattr(field, 'data_type', None)
                if dt and ('Text' in dt or 'String' in dt):
                    if field.column in df.columns:
                        group_field_id = field.column
                        break

    print(f"Selected numeric field: {numeric_field_id}")
    print(f"Selected group field: {group_field_id}")

    # Only continue if found
    if numeric_field_id:
        # Coerce to numeric if not already
        df_copy = df.copy()
        df_copy[numeric_field_id] = pd.to_numeric(df_copy[numeric_field_id], errors='coerce')
        threshold = df_copy[numeric_field_id].quantile(0.90)  # use 90th percentile as an example threshold
        filtered_df = df_copy[df_copy[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        display(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean())/filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Grouping (if possible)
        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped mean {numeric_field_id} by {group_field_id}:")
            display(grouped_df.head())
        else:
            print("No suitable group field found for grouping.")
    else:
        print("No suitable numeric field found.")
else:
    print("No extracted record set DataFrame available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. All fields/columns are referenced by their Croissant `@id`s.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_rs_id and numeric_field_id:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.xlabel(f'Field (@id): {numeric_field_id}')
    plt.title(f'Distribution of {numeric_field_id}')
    plt.show()

    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.xlabel(f'Group field (@id): {group_field_id}')
        plt.ylabel(f'Numeric field (@id): {numeric_field_id}')
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Not enough information to generate visualizations (need valid numeric field and data).")

## 6. Conclusion
This notebook demonstrated:
- How to load and explore a Croissant-described dataset using `mlcroissant`.
- Programmatic identification of record sets, fields, columns, and referencing them via their `@id`s.
- Example EDA processes such as filtering, normalization, and grouping by categorical fields.
- Visualizing distributions and grouped summaries.

Further analysis may include deeper domain-specific feature engineering, advanced visualizations, and downstream data science tasks.
